# Electricity Theft Detection

**Track:** Energy Systems — Predictive Maintenance / Anomaly Detection
**Advanced Topics:** XAI (SHAP) + Adversarial Robustness (FGSM)
**Dataset:** UCI Electricity Load Diagrams + Synthetic Theft Labels

In [ ]:
import subprocess
subprocess.check_call(['pip', 'install', 'shap'])
print('Dependencies installed')
# NOTE: This cell requires a Colab runtime restart after execution before proceeding to next cells.

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import shap
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, classification_report, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch: {torch.__version__}')
print(f'SHAP: {shap.__version__}')

In [ ]:
# Install kagglehub for reliable dataset access
import subprocess
subprocess.check_call(['pip', 'install', '-q', 'kagglehub'])
print('kagglehub installed')

# Download Electricity Load Diagrams dataset via kagglehub
import kagglehub
path = kagglehub.dataset_download("eduardojst10/electricityloaddiagrams20112014")

# Read only the first meter (Meter_001) to keep Colab memory manageable
import pandas as pd
meter_path = f"{path}/Electricity/Meter_001.csv"
df = pd.read_csv(meter_path)
print(f'Shape: {df.shape}')
print(df.head())
print(df.info())

In [ ]:
# Plot load data for the first meter
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
for i in range(min(4, len(df))):
    # Show a subset of data for each subplot (different time ranges)
    start = i * len(df) // 4
    end = start + len(df) // 8
    axes[i].plot(range(start, end), df.iloc[start:end].values, linewidth=0.5)
    axes[i].set_title(f'Portion {i+1} of Meter_001 Load Diagram')
    axes[i].set_ylabel('Load (kWh)')
plt.tight_layout()
plt.savefig('baseline_load_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

# Basic statistics
print(df.describe())

# Check for missing values
missing = df.isnull().sum()
print(f'Missing values per column:\n{missing[missing > 0]}')

In [ ]:
def create_time_windows(data, window_size=24):
    """
    Convert raw load data into sliding time windows of 'window_size' hours.
    Returns arrays of sequences and features (hour-of-day encoding).
    """
    X_sequences = []
    for i in range(0, len(data) - window_size + 1):
        window = data[i:i + window_size]
        hour_features = np.zeros(window_size)
        # Encode time of day: sin/cos encoding for cyclical patterns
        hours = np.arange(window_size)
        hour_features[:] = np.sin(2 * np.pi * hours / 24)
        X_sequences.append({
            'load_window': window.values.reshape(-1, 1),
            'hour_encoding': hour_features
        })

    load_data = np.stack([x['load_window'].flatten() for x in X_sequences])
    hour_data = np.stack([x['hour_encoding'] for x in X_sequences])
    return load_data, hour_data

# Apply to first meter
meter_col = df.columns[0]
raw_series = df[meter_col].ffill().bfill().values
X_load, X_hour = create_time_windows(raw_series, window_size=24)
print(f'Windowed shape: {X_load.shape}')  # (n_samples, 24)

In [ ]:
def generate_synthetic_labels(X_load, n_anomalies=0.15, seed=42):
    """
    Generate synthetic electricity theft labels by injecting anomalies into normal load data.
    Three types of theft patterns:
    - Sudden drops (bypassed meter): reduce consumption 50-80%
    - Unusual spikes (illegal connections): add 2-3x normal baseline
    - Flat-line periods (tampered reading): set to zero or constant

    Returns corrupted load array and binary labels.
    """
    np.random.seed(seed)
    n_samples = len(X_load)
    labels = np.zeros(n_samples, dtype=int)
    X_corrupted = X_load.copy()

    n_anom = int(n_samples * n_anomalies)
    anom_indices = np.random.choice(n_samples, n_anom, replace=False)

    anomaly_types = np.random.randint(0, 3, n_anom)

    for idx, atype in zip(anom_indices, anomaly_types):
        if atype == 0:  # Sudden drop (bypassed meter)
            reduction = np.random.uniform(0.5, 0.8)
            X_corrupted[idx] *= reduction
            labels[idx] = 1
        elif atype == 1:  # Unusual spike (illegal connection)
            multiplier = np.random.uniform(2.0, 3.0)
            baseline_mean = X_load[idx].mean()
            X_corrupted[idx] += baseline_mean * (multiplier - 1)
            labels[idx] = 1
        else:  # Flat-line (tampered reading)
            constant_val = np.random.uniform(0, X_load[idx].max() * 0.1)
            X_corrupted[idx] = constant_val
            labels[idx] = 1

    return X_corrupted, labels

X_thief, y_labels = generate_synthetic_labels(X_load, n_anomalies=0.15)
print(f'Anomaly rate: {y_labels.mean():.2%}')

In [ ]:
# Combine load windows and hour encodings into single feature vector
X_combined = np.concatenate([X_thief, X_hour], axis=1)  # (n_samples, 48)

# Temporal train/test split (do NOT shuffle — simulate real-world deployment)
split_idx = int(len(X_combined) * 0.8)
X_train_raw = X_combined[:split_idx]
X_test_raw = X_combined[split_idx:]
y_train = y_labels[:split_idx]
y_test = y_labels[split_idx:]

# Feature normalization (fit on train only, apply to both)
mean = X_train_raw.mean(axis=0)
std = X_train_raw.std(axis=0) + 1e-8
X_train = (X_train_raw - mean) / std
X_test = (X_test_raw - mean) / std

# Convert to PyTorch tensors
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.LongTensor(y_train)
X_test_t = torch.FloatTensor(X_test)
y_test_t = torch.LongTensor(y_test)

train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

print(f'Train: {len(train_dataset)} samples ({y_train.sum()} anomalies)')
print(f'Test:  {len(X_test_t)} samples ({y_test_t.sum().item()} anomalies)')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(['Normal', 'Theft'], [y_train.sum(), len(y_train) - y_train.sum()], color=['green', 'red'])
axes[0].set_title('Training Set Distribution')

axes[1].bar(['Normal', 'Theft'], [y_test.sum(), len(y_test) - y_test.sum()], color=['green', 'red'])
axes[1].set_title('Test Set Distribution')
plt.tight_layout()
plt.savefig('label_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
class TheftDetectionLSTM(nn.Module):    def __init__(self, input_size=48, hidden_size=64, num_layers=2, dropout=0.2):        super(TheftDetectionLSTM, self).__init__()        self.hidden_size = hidden_size        self.num_layers = num_layers        self.lstm = nn.LSTM(            input_size=input_size,            hidden_size=hidden_size,            num_layers=num_layers,            batch_first=True,            dropout=dropout if num_layers > 1 else 0.0        )        self.classifier = nn.Sequential(            nn.Linear(hidden_size, 32),            nn.ReLU(),            nn.Dropout(dropout),            nn.Linear(32, 1),            nn.Sigmoid()        )    def forward(self, x):        # x shape: (batch, seq_len, features)        lstm_out, (h_n, c_n) = self.lstm(x)        # Use last hidden state for classification        out = self.classifier(h_n[-1])        return out.squeeze()model = TheftDetectionLSTM(input_size=48, hidden_size=64, num_layers=2)print(model)print(f'Total parameters: {sum(p.numel() for p in model.parameters())}')

In [ ]:
# Compute class weights for imbalanced datapos_weight = torch.tensor([(len(y_train) - y_train.sum()) / max(y_train.sum(), 1)])criterion = nn.BCELoss(weight=pos_weight)optimizer = torch.optim.Adam(model.parameters(), lr=0.001)epochs = 30train_losses = []for epoch in range(epochs):    model.train()    epoch_loss = 0    n_batches = 0    for batch_X, batch_y in train_loader:        optimizer.zero_grad()        outputs = model(batch_X)        loss = criterion(outputs, batch_y.float())        loss.backward()        optimizer.step()        epoch_loss += loss.item()        n_batches += 1    avg_loss = epoch_loss / n_batches    train_losses.append(avg_loss)    if (epoch + 1) % 5 == 0:        print(f'Epoch [{epoch+1}/{epochs}], Loss: {avg_loss:.4f}')# Save trained modeltorch.save(model.state_dict(), 'theft_detection_lstm.pth')plt.figure(figsize=(10, 4))plt.plot(train_losses)plt.title('Training Loss Over Epochs')plt.xlabel('Epoch'); plt.ylabel('Loss')plt.savefig('training_loss.png', dpi=150, bbox_inches='tight')plt.show()print('Model saved to theft_detection_lstm.pth')

In [ ]:
# Evaluate on test setmodel.eval()all_preds = []all_probs = []with torch.no_grad():    for i in range(0, len(X_test_t), 64):        batch = X_test_t[i:i+64]        probs = model(batch)        all_probs.extend(probs.numpy())        preds = (probs >= 0.5).long()        all_preds.extend(preds.numpy())all_preds = np.array(all_preds)all_probs = np.array(all_probs)# Classification reportprecision, recall, f1, _ = precision_recall_fscore_support(y_test_t, all_preds, average='binary')print(f'Precision: {precision:.4f}')print(f'Recall:    {recall:.4f}')print(f'F1-Score:  {f1:.4f}')print(f'\nClassification Report:\n{classification_report(y_test_t, all_preds, target_names=["Normal", "Theft"])}')